# Outlier Detection: Z-Score and IQR Methods

Detect anomalies using statistical methods without sklearn. Manual implementation for educational purposes.

## 1. Setup and Data Generation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '../')

from src import statistical_analysis, feature_engineering

np.random.seed(42)

# Generate network traffic data
normal_duration = np.random.exponential(500, 1000)  # Normal
attack_duration = np.random.exponential(50, 50)      # Attack anomalies
data = np.concatenate([normal_duration, attack_duration])

print(f"Dataset: {len(data)} records")
print(f"Normal: {len(normal_duration)}, Anomalies: {len(attack_duration)}")
print(f"\nStatistics:")
print(f"  Mean: {data.mean():.2f}")
print(f"  Std:  {data.std():.2f}")
print(f"  Min:  {data.min():.2f}")
print(f"  Max:  {data.max():.2f}")

## 2. Manual Z-Score Computation

In [ ]:
# Z-Score = (X - Mean) / Std

series = pd.Series(data)
mean = series.mean()
std = series.std()

# Manual computation
z_scores_manual = (data - mean) / std

# Using src function
z_scores = statistical_analysis.compute_zscore(series)

print("Z-Score Examples (first 20):")
df_z = pd.DataFrame({
    'Value': data[:20],
    'Z-Score': z_scores[:20],
    'Abs_Z': np.abs(z_scores[:20])
})
print(df_z.to_string())

print(f"\nZ-Score Statistics:")
print(f"  Mean: {z_scores.mean():.4f} (should be ~0)")
print(f"  Std:  {z_scores.std():.4f} (should be ~1)")
print(f"  Max:  {z_scores.max():.4f}")

## 3. Z-Score Anomaly Detection (Threshold = 3)

In [ ]:
# 3-sigma rule: ~99.7% of data is within 3 standard deviations
# Points with |Z| > 3 are highly anomalous

threshold = 3.0
anomalies_zscore = statistical_analysis.zscore_anomaly_detection(data, threshold=threshold)

print(f"Z-Score Anomaly Detection (threshold={threshold}):")
print(f"  Detected: {anomalies_zscore.sum()} anomalies out of {len(data)}")
print(f"  Detection rate: {anomalies_zscore.sum()/len(data)*100:.2f}%")

# View detected anomalies
anomaly_indices = np.where(anomalies_zscore)[0]
print(f"\nAnomalous values (top 10):")
print(data[anomaly_indices][:10])

# Classification
classification = pd.DataFrame({
    'Value': data[:50],
    'Z_Score': z_scores[:50],
    'Is_Anomaly': anomalies_zscore[:50]
})
print("\nFirst 50 records:")
print(classification.to_string())

## 4. Manual IQR Computation

In [ ]:
# IQR = Interquartile Range = Q3 - Q1
# Outliers: < (Q1 - 1.5*IQR) or > (Q3 + 1.5*IQR)

series = pd.Series(data)
Q1 = series.quantile(0.25)
Q3 = series.quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

print("IQR Analysis:")
print(f"  Q1 (25th percentile): {Q1:.2f}")
print(f"  Q3 (75th percentile): {Q3:.2f}")
print(f"  IQR: {IQR:.2f}")
print(f"  Lower bound: {lower_bound:.2f}")
print(f"  Upper bound: {upper_bound:.2f}")
print(f"\nNormal range: [{lower_bound:.2f}, {upper_bound:.2f}]")

## 5. IQR Anomaly Detection

In [ ]:
anomalies_iqr = statistical_analysis.iqr_anomaly_detection(series, multiplier=1.5)

print(f"IQR Anomaly Detection (multiplier=1.5):")
print(f"  Detected: {anomalies_iqr.sum()} anomalies out of {len(data)}")
print(f"  Detection rate: {anomalies_iqr.sum()/len(data)*100:.2f}%")

# Anomalous values
anomaly_values_iqr = data[anomalies_iqr.values]
print(f"\nAnomalous values (first 20):")
print(anomaly_values_iqr[:20])

# Statistics of anomalies
print(f"\nAnomalies statistics:")
print(f"  Count: {len(anomaly_values_iqr)}")
print(f"  Mean: {anomaly_values_iqr.mean():.2f}")
print(f"  Min: {anomaly_values_iqr.min():.2f}")
print(f"  Max: {anomaly_values_iqr.max():.2f}")

## 6. Comparison: Z-Score vs IQR

In [ ]:
# Compare detection methods
comparison = pd.DataFrame({
    'Method': ['Z-Score (|Z|>3)', f'IQR (1.5x)'],
    'Detected': [anomalies_zscore.sum(), anomalies_iqr.sum()],
    'Detection_Rate_%': [
        anomalies_zscore.sum() / len(data) * 100,
        anomalies_iqr.sum() / len(data) * 100
    ]
})

print("\nComparison of Methods:")
print(comparison.to_string(index=False))

# Agreement between methods
agreement = (anomalies_zscore == anomalies_iqr).sum()
print(f"\nBoth methods agree on: {agreement} records ({agreement/len(data)*100:.1f}%)")
print(f"Disagreement: {len(data) - agreement} records ({(len(data)-agreement)/len(data)*100:.1f}%)")

## 7. Visualization: Threshold Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('Outlier Detection: Z-Score vs IQR', fontsize=14, fontweight='bold')

# Z-Score visualization
axes[0, 0].scatter(range(len(data)), data, alpha=0.5, s=10, label='Data')
axes[0, 0].scatter(np.where(anomalies_zscore)[0], data[anomalies_zscore], 
                   color='red', s=50, label='Anomalies', marker='^')
axes[0, 0].set_ylabel('Flow Duration')
axes[0, 0].set_title(f'Z-Score Detection (|Z| > 3)')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Z-Score histogram
axes[0, 1].hist(z_scores, bins=50, alpha=0.7, edgecolor='black')
axes[0, 1].axvline(-3, color='red', linestyle='--', linewidth=2, label='±3σ threshold')
axes[0, 1].axvline(3, color='red', linestyle='--', linewidth=2)
axes[0, 1].set_xlabel('Z-Score')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Z-Score Distribution')
axes[0, 1].legend()
axes[0, 1].grid(axis='y', alpha=0.3)

# IQR visualization
axes[1, 0].scatter(range(len(data)), data, alpha=0.5, s=10, label='Data')
axes[1, 0].axhline(lower_bound, color='orange', linestyle='--', linewidth=2, label='IQR Bounds')
axes[1, 0].axhline(upper_bound, color='orange', linestyle='--', linewidth=2)
axes[1, 0].scatter(np.where(anomalies_iqr)[0], data[anomalies_iqr.values], 
                   color='red', s=50, label='Anomalies', marker='^')
axes[1, 0].set_ylabel('Flow Duration')
axes[1, 0].set_title(f'IQR Detection (1.5x multiplier)')
axes[1, 0].legend()
axes[1, 0].grid(alpha=0.3)

# Boxplot showing IQR
bp = axes[1, 1].boxplot(data, vert=True, patch_artist=True)
bp['boxes'][0].set_facecolor('lightblue')
axes[1, 1].scatter(np.random.normal(1, 0.04, np.sum(anomalies_iqr)), 
                  data[anomalies_iqr.values], color='red', s=50, alpha=0.7, label='Outliers')
axes[1, 1].set_ylabel('Flow Duration')
axes[1, 1].set_title('Boxplot: IQR Visualization')
axes[1, 1].legend()
axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Multivariate Detection (Multiple Features)

In [ ]:
# Create multi-feature dataset
n = 100
normal = {
    'Duration': np.random.exponential(500, int(n*0.9)),
    'Packets': np.random.poisson(10, int(n*0.9)),
    'Bytes': np.random.normal(2000, 500, int(n*0.9))
}

anomaly = {
    'Duration': np.random.exponential(50, int(n*0.1)),
    'Packets': np.random.poisson(200, int(n*0.1)),
    'Bytes': np.random.normal(50000, 10000, int(n*0.1))
}

df_normal = pd.DataFrame(normal)
df_anomaly = pd.DataFrame(anomaly)
df_multi = pd.concat([df_normal, df_anomaly], ignore_index=True)

# Multi-feature detection
anomalies_multi = statistical_analysis.multivariate_iqr_detection(df_multi, multiplier=1.5)

print(f"Multivariate IQR Detection:")
print(f"  Total records: {len(df_multi)}")
print(f"  Anomalies: {anomalies_multi.sum()}")
print(f"  Detection rate: {anomalies_multi.sum()/len(df_multi)*100:.1f}%")

# Show results
df_multi['Is_Anomaly'] = anomalies_multi
print("\nLast 20 records:")
print(df_multi.tail(20).to_string())

## Summary

### Z-Score Method
- **Formula**: Z = (X - mean) / std
- **Threshold**: Usually |Z| > 3 (99.7% of data)
- **Pros**: Simple, fast, good for normal distributions
- **Cons**: Sensitive to extreme outliers

### IQR Method
- **Formula**: Outliers outside [Q1 - 1.5×IQR, Q3 + 1.5×IQR]
- **Advantage**: Robust to extreme values
- **Pros**: Doesn't assume normality
- **Cons**: May miss subtle anomalies

### Best Practice
- **Use both methods** and ensemble results
- Weighted combination catches more anomalies
- No single method is perfect!

SENTINEL uses **weighted combination** of both methods for best results.